# Activations Manager

This notebook includes cells to perform different kinds of data manipulation with the activations

## Import libraries

In [2]:
from activations import *

from utils import ACTIVATIONS_FOLDER, MODELS_FOLDER, LANGUAGES, SPLITS, MODEL_NAMES
from icecream import ic

device: t.device = t.device("cuda" if t.cuda.is_available() else "cpu")
device

device(type='cuda')

## Activation Generator

In [2]:
save_to_disk: bool = True
amount_of_batches_to_generate: int | None = None
batch_size: int = 128
# model_name: str = "tiny_aya_global"
languages_to_generate: list[str] = LANGUAGES
splits_to_generate: list[str] = SPLITS

debug = False
if debug:
    languages_to_generate = ["en"]
    languages_to_generate = LANGUAGES
    splits_to_generate = ["train"]
    amount_of_batches_to_generate = 2
    batch_size = 4
    print("Running in debug mode")

ic(save_to_disk, amount_of_batches_to_generate, batch_size, languages_to_generate, splits_to_generate)


assert save_to_disk, ("If generating, must also save to disk")

ic| save_to_disk: True
    amount_of_batches_to_generate: None
    batch_size: 128
    languages_to_generate: ['en', 'es', 'jp']
    splits_to_generate: ['train', 'test', 'val']


In [4]:
languages_to_generate = ["jp"]
batch_size = 64

model_name: str = "olmo_model"

activation_recorder: ActivationRecorder = ActivationRecorder(model_name)

special_cases: list[SpecialCase] = []    

activation_recorder.generate_all_activations(
    languages_to_generate,
    splits_to_generate,
    amount_of_batches_to_generate,
    save_to_disk,
    batch_size,
    special_cases=special_cases,
)

--------------------
Generating language jp, split train
--------------------
The tokenizer or model were not loaded for the ActivationLoader of olmo_model. Loading now.


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 784.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 30.68 GiB is allocated by PyTorch, and 7.71 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
languages_to_generate = LANGUAGES

model_name: str = "tiny_aya_global"

activation_recorder: ActivationRecorder = ActivationRecorder(model_name)

special_cases: list[SpecialCase] = []    

activation_recorder.generate_all_activations(
    languages_to_generate,
    splits_to_generate,
    amount_of_batches_to_generate,
    save_to_disk,
    batch_size,
    special_cases=special_cases,
)

## Sanity Checks

In [4]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = True
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    splits = ["train"]
    print(f"Using custom configuration")
    ic(model_names, languages, splits)

for model_name in model_names:
    for language in languages:
        for split in splits:
            activations_dataset: ActivationDataset = ActivationDataset(language, split, 0, "standard", model_name)

            print(activations_dataset.activations.shape)
            for i in range(8):
                print(activations_dataset[i][0])
                print(activations_dataset[i][1])
                print("\n-----------------")

ic| 

model_names: ['olmo_model'], languages: ['en'], splits: ['train']


Using custom configuration
torch.Size([4439, 4096])
tensor([-0.0449, -0.0151, -0.0144,  ...,  0.0273, -0.0610,  0.0527],
       dtype=torch.bfloat16)
tensor(1, dtype=torch.int32)

-----------------
tensor([ 0.0004, -0.0083,  0.0088,  ..., -0.0081,  0.0356, -0.0216],
       dtype=torch.bfloat16)
tensor(1, dtype=torch.int32)

-----------------
tensor([-0.0249, -0.0223,  0.0015,  ...,  0.0175, -0.0718,  0.0464],
       dtype=torch.bfloat16)
tensor(0, dtype=torch.int32)

-----------------
tensor([-0.0267, -0.0237, -0.0077,  ...,  0.0277, -0.0635,  0.0613],
       dtype=torch.bfloat16)
tensor(1, dtype=torch.int32)

-----------------
tensor([-0.0347, -0.0208, -0.0121,  ...,  0.0347, -0.0649,  0.0576],
       dtype=torch.bfloat16)
tensor(1, dtype=torch.int32)

-----------------
tensor([-0.0154, -0.0260, -0.0503,  ...,  0.0083, -0.0444,  0.0376],
       dtype=torch.bfloat16)
tensor(1, dtype=torch.int32)

-----------------
tensor([-0.0282, -0.0228, -0.0193,  ...,  0.0317, -0.0615,  0.0479],
   

## Activation Merger

In [3]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = False
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    splits = ["train", "test"]
    print("Using custom configuration")
else:
    print("Using default configuration")

ic(model_names, languages, splits, custom)

for model_name in model_names:
    num_layers: int = ActivationRecorder(model_name).get_number_of_layers()
    for language in languages:
        for split in splits:
            try:
                for layer_num in range(num_layers):
                    print(f"Merging {model_name}/{language}/{split}")
                    merge_activation_batches(model_name, language, split, layer_num)
            except FileNotFoundError as e:
                print(f"Couldn't merge activations of {model_name}, {language}, {split} due to:\n {e}")
                print("It is possible that these activations were already merged")

Using default configuration


ic| model_names: ['olmo_model', 'tiny_aya_global']
    languages: ['en', 'es', 'jp', 'nl']
    splits: ['train', 'test', 'val']
    custom: False


Model not loaded. Getting the number of layers from ./data/activations/olmo_model/n_layers.txt
Merging olmo_model/en/train
Merged 35 batch files for olmo_model, en, train, layer0
Merged activations shape: torch.Size([4439, 4096])
Saved to ./data/activations/olmo_model/en/train/layer0_merged.pt
Merging olmo_model/en/train
Merged 35 batch files for olmo_model, en, train, layer1
Merged activations shape: torch.Size([4439, 4096])
Saved to ./data/activations/olmo_model/en/train/layer1_merged.pt
Merging olmo_model/en/train
Merged 35 batch files for olmo_model, en, train, layer2
Merged activations shape: torch.Size([4439, 4096])
Saved to ./data/activations/olmo_model/en/train/layer2_merged.pt
Merging olmo_model/en/train
Merged 35 batch files for olmo_model, en, train, layer3
Merged activations shape: torch.Size([4439, 4096])
Saved to ./data/activations/olmo_model/en/train/layer3_merged.pt
Merging olmo_model/en/train
Merged 35 batch files for olmo_model, en, train, layer4
Merged activations sh

## Activation Deleter

In [3]:
# models_to_delete: list[str] = ["olmo_model"]
# languages_to_delete: list[str] = ["es"]
models_to_delete: list[str] = MODEL_NAMES
languages_to_delete: list[str] = LANGUAGES
splits_to_delete: list[str] = SPLITS

actually_delete = True

for model_name in models_to_delete:
    for language in languages_to_delete:
        for split in splits_to_delete:
            delete_activations_file(model_name, language, split, layer_num=None, batch_id=None, ignore_substring="batch", actually_delete=actually_delete)


Deleted data\activations\olmo_model\en\train\layer0_merged.pt
Deleted data\activations\olmo_model\en\train\layer10_merged.pt
Deleted data\activations\olmo_model\en\train\layer11_merged.pt
Deleted data\activations\olmo_model\en\train\layer12_merged.pt
Deleted data\activations\olmo_model\en\train\layer13_merged.pt
Deleted data\activations\olmo_model\en\train\layer14_merged.pt
Deleted data\activations\olmo_model\en\train\layer15_merged.pt
Deleted data\activations\olmo_model\en\train\layer16_merged.pt
Deleted data\activations\olmo_model\en\train\layer17_merged.pt
Deleted data\activations\olmo_model\en\train\layer18_merged.pt
Deleted data\activations\olmo_model\en\train\layer19_merged.pt
Deleted data\activations\olmo_model\en\train\layer1_merged.pt
Deleted data\activations\olmo_model\en\train\layer20_merged.pt
Deleted data\activations\olmo_model\en\train\layer21_merged.pt
Deleted data\activations\olmo_model\en\train\layer22_merged.pt
Deleted data\activations\olmo_model\en\train\layer23_merg

In [ ]:
models_to_delete: list[str] = MODEL_NAMES
languages_to_delete: list[str] = LANGUAGES
splits_to_delete: list[str] = SPLITS

actually_delete = False

for model_name in models_to_delete:
    for language in languages_to_delete:
        for split in splits_to_delete:
            delete_activations_file(model_name, language, split, layer_num=None, batch_id=None, ignore_substring="merged", actually_delete=actually_delete)

No files found matching the criteria in data\activations\olmo_model\nl\train
No files found matching the criteria in data\activations\olmo_model\nl\test
Directory not found: data\activations\olmo_model\nl\val
Directory not found: data\activations\tiny_aya_global\nl\train
Directory not found: data\activations\tiny_aya_global\nl\test
Directory not found: data\activations\tiny_aya_global\nl\val
